<img src="images/logodwengo.png" alt="LogoDwengo" width="150"/>

<div>
    <font color=#690027 markdown="1">
<h1>SIMULEZ UNE ÉPIDÉMIE: UNE FLAMBÉE DE MALADIE DANS UN RÉSEAU SOCIAL</h1>    </font>
</div>

<div class="alert alert-box alert-success">
Dans ce projet, tu étudies comment les maladies peuvent se propager à travers un réseau (social). Tu examines comment la structure d'un réseau peut influencer la vitesse à laquelle une maladie est transmise. Enfin, tu examineras également différentes stratégies pour lutter contre la propagation d'une maladie.<br>Dans ce notebook, vous appliquez le modèle SIR au sein d'un réseau social.</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.spatial import distance_matrix

## Une épidémie dans un réseau social

Regardez maintenant comment vous pouvez traduire le modèle SIR de propagation des maladies dans le langage des réseaux. <br>À l'aide d'un réseau général, vous allez établir un modèle beaucoup plus réaliste. Plus d'approximation continue ! Ce modèle correspond étonnamment mieux à la réalité et, de plus, il est beaucoup plus simple à comprendre et à simuler. Vous pouvez obtenir une solution exacte sans avoir besoin de dérivées ni d'autres techniques mathématiques avancées !
### Dynamique de la maladie sur un réseau
Au lieu de suivre le nombre d’individus $S$, $I$ et $R$ au cours du temps comme dans le modèle SIR standard, tu conserveras pour chaque nœud du réseau son état. Le temps ne variera pas de manière continue mais s’écoulera maintenant en pas discrets : $t = 0, 1, 2, 3, \ldots$. <br>- L’état du nœud numéro $i$ au temps $t$ est décrit par $N_i^t\in \{S, I, R\}$. Cela signifie que le nœud $i$ au temps $t$ peut être dans l’état $S$ (susceptible), $I$ (infecté) ou $R$ (résistant).- Le changement d'état des nœuds se décrit au moyen de quelques règles simples. À l'instar du modèle SIR original, qui comporte deux paramètres, beta (le taux d'infection) et gamma (le taux de guérison), le modèle SIR pour un réseau comporte lui aussi deux paramètres.

#### Personnes susceptibles et infectées
Vous vous limitez d’abord aux individus susceptibles et infectés. Vous partez du principe que les individus susceptibles peuvent être infectés et que les individus infectés peuvent devenir résistants. Il n’y a donc pas de transition possible d’infecté à susceptible, ni de susceptible à résistant. Considérez les règles suivantes:
- Si un nœud se trouve au temps $t$ dans l'état $S$, alors chaque voisin **infecté** a une probabilité $p_\text{inf}$ de transmettre la maladie. Le nœud passe à l'état $I$ si au moins un voisin transmet la maladie.- Si un nœud est dans l'état $I$ à l'instant $t$, alors il passe à l'état $R$ avec une probabilité $p_\text{res}$.

Donc, supposons qu'un nœud soit dans l'état $S$, et qu'il ait $k$ voisins qui sont dans l'état $I$. La probabilité qu'aucun voisin ne transmette la maladie est alors :
$$(1-p_\text{inf})^k,$$
donc la probabilité que la maladie soit effectivement transmise, et qu'il y ait donc une transition de l'état $S$ vers $I$, est :
$$1 - (1-p_\text{inf})^k\,.$$
Tu as utilisé ici la règle du produit et la règle du complément en calcul des probabilités.


#### ExempleConsidérez le nœud encadré en bleu dans la figure ci-dessous. Supposons que $p_\text{inf}=0.2$, quelle est la probabilité que l’un des trois voisins malades transmette la maladie ?
![](images/ziekteverspr.png)<center> Figure 1.</center>
Vous le calculez avec le code suivant :

In [ ]:
p_inf = 0.2
k = 3

p_ziekte_doorgegeven = 1 - (1 - p_inf)**k

print("Kans om de ziekte te krijgen is:", p_ziekte_doorgegeven)

Vous pouvez simuler la transmission effective de la maladie avec NumPy, où `np.random.rand()` génère un nombre aléatoire uniformément distribué entre 0 et 1. <br>Vous le faites avec le code dans la cellule de code suivante. Pour la simulation, exécutez cette cellule plusieurs fois.

In [ ]:
# voorbeeld
p_ziekte_doorgegeven > np.random.rand()

In [ ]:
# voorbeeld
p_ziekte_doorgegeven > np.random.rand()

In [ ]:
# voorbeeld
p_ziekte_doorgegeven > np.random.rand()

Avec `True`, la maladie est effectivement transmise, avec `False` non. Notez qu'un facteur aléatoire est intégré dans la simulation.

> **Exercice 1**: Supposons que $p_\text{inf}=1$ (toute personne malade transmet immédiatement la maladie à tous ses voisins dans le réseau). Initialement, seuls les nœuds 1 et 11 sont infectés dans le réseau d'exemple de la Figure 3 du notebook précédent sur les réseaux sociaux.<br>-  Qui est infecté à l'étape suivante?-  Et à l'étape suivante ?

Réponse:

### ImplémentationVous pouvez implémenter le modèle facilement en Python à l'aide de SciPy. <br>Vous commencerez par générer un simple réseau social pour illustrer ce modèle :-  Vous générez pour cela une population de `n` personnes. Pour garder un aspect visuel, celles-ci sont représentées par des points dans le plan $x,y$.- Par la suite, vous générez une matrice d'adjacence qui indique s'il existe une connexion entre les nœuds.

#### D'abord, vous générez les nœuds du réseau. Simultanément, vous générez la distance entre les nœuds.

In [ ]:
def genereer_populatie(n):
    """Genereren van punten en bepalen van hun onderlinge afstand."""
    # n punten genereren, uniform in het xy-vlak
    X = np.random.rand(n, 2)
    # alle paarsgewijze afstanden tussen n punten
    D = distance_matrix(X, X)
    return X, D

In [ ]:
# populatie van netwerk van 200 punten genereren
n = 200
X, D = genereer_populatie(n)

In [ ]:
print(X,D)

Les distances entre deux personnes constituent la matrice de distances $D$.

In [ ]:
# X bestaat uit 200 koppels en D is 200x200-matrix
print(X.shape, D.shape)

#### Générez maintenant la matrice de connectivité V.

Pour obtenir un modèle simple pour la matrice de connectivité V, on suppose que la probabilité que $v_{ij}=1$, c’est-à-dire que les nœuds $i$ et $j$ sont connectés, est donnée par :
$$p_{ij} = \exp(-\alpha \, d_{ij})\,.$$
**Ici, la probabilité d’une connexion entre les nœuds $i$ et $j$ diminue à mesure que la distance entre les deux nœuds augmente.** <br>$\alpha$ est un paramètre ($\alpha \geq 0$) qui régit cette relation. Une grande valeur de $\alpha$ fait en sorte que deux nœuds très éloignés ont une très faible probabilité d'être reliés. Pour une petite valeur de $\alpha$, cela reste possible. De plus, plus la distance entre deux nœuds est grande, plus la probabilité d'une connexion est faible.

In [ ]:
# illustratie van effect van waarde van alpha
plt.figure() 

xwaarden = np.linspace(0, 10, 100)
plt.plot(xwaarden, np.exp(-0.1 * xwaarden), label=r"$\alpha=0.1$")       # r in omschrijving label omwille van LaTeX-code
plt.plot(xwaarden, np.exp(-0.5 * xwaarden), label=r"$\alpha=0.5$")
plt.plot(xwaarden, np.exp(-1 * xwaarden), label=r"$\alpha=1$")
plt.plot(xwaarden, np.exp(-5 * xwaarden), label=r"$\alpha=5$")
plt.plot(xwaarden, np.exp(-10 * xwaarden), label=r"$\alpha=10$")
plt.xlabel(r"Afstand $d_{ij}$")                 
plt.ylabel(r"Kans op verbinding $v_{ij}$")
plt.legend(loc=0)

plt.show()

> **Exercice 2**: Réfléchissez bien à la signification de $\alpha$. Que se passe-t-il si $\alpha=0$ ? Que se passe-t-il si $\alpha$ est très grand ?

Réponse:

In [ ]:
def sample_verbindingsmatrix(D, alpha=1.0):
    """Genereren van verbindingsmatrix afhankelijk van afstandsmatrix en alpha."""
   
    # verbindingsmatrix heeft dezelfde dimensie als afstandsmatrix, beide zijn vierkant
    n = D.shape[1]             # aantal kolommen in D is gelijk aan populatiegrootte
    
    # matrix aanmaken met 0 en 1 om verbindingen voor te stellen
    # alle elementen op diagonaal zijn nul, matrix is symmetrisch
    A = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i+1, n):
                 # kans op een verbinding
                 p = np.exp(- alpha * D[i,j])
                 # met een kans van p, maak een verbinding tussen i en j
                 if p > np.random.rand():
                        A[i,j] = 1
                        A[j,i] = 1      # symmetrische matrix
    return A


In [ ]:
# verbindingsmatrix van netwerk genereren voor alpha = 10
V = sample_verbindingsmatrix(D, alpha=10)
print(V)        # elke matrix kan gebruikt worden om figuur te representeren  
print(V.min(), V.max())

In [ ]:
# visualiseren dat V uit nullen en enen bestaat
plt.imshow(V, cmap="gray")   # elke matrix kan gebruikt worden als representatie voor afbeelding, 0 zwart, 1 wit 

#### Représenter le réseau par un graphe.

Pour cela, tu écris une nouvelle fonction en Python.<br> Les personnes infectées seront affichées en rouge, les résistantes en vert et les susceptibles en jaune. Tu utiliseras donc un graphe coloré. <br>Si l'état des nœuds n'a pas encore été transmis, colorez-les en bleu.
La liste des points (nœuds) du réseau s’accompagne donc également d’une liste d’états, le premier état correspondant au premier nœud, le deuxième état au deuxième nœud, etc.

In [ ]:
 def plot_netwerk(X, V, toestanden=None):
    """Graaf van het netwerk.""" 
    n = V.shape[1]          # populatiegrootte is gelijk aan aantal kolommen van V
    
    # van elke knoop kleur nagaan en lijst van maken
    if toestanden is None:
        # geen toestanden gegeven, alle knopen zijn blauw
        knoop_kleuren = "blue"
    else:
        kleur_map = {"S" : "yellow", "I" : "red", "R" : "green"}    # dictionary
        knoop_kleuren = [kleur_map[toestand] for toestand in toestanden]
    
    
    plt.figure(figsize=(15,10))
    
    plt.axis("off")  # bij graaf geen assen  
    
    # plot n knopen, eerste kolom van X bevat x-coördinaat, tweede kolom y-coördinaat in juiste kleur
    plt.scatter(X[:,0], X[:,1], color=knoop_kleuren, zorder=1)    # zorder=1: punten op bovenste layer van graaf
    
    # teken verbindingen in grijs
    # n is populatiegrootte en V[i,j] is waarde van verbinding (0 of 1)
    # als V[i,j] = 1, dan lijnstuk tussen i-de en j-de knoop
    # plot om i-de en j-de knoop te verbinden
    # i-de en j-de knoop staan op i-de en j-de rij van X, dus X[i,j] nodig met x'n in eerste kolom daarvan en y's in tweede
    for i in range(n):
        for j in range(i+1, n):
            if V[i,j] == 1:
                plt.plot(X[[i,j],0], X[[i,j],1], alpha=0.8, color="grey", zorder=0)    # zorder=0: lijnen onderste layer van graaf
    plt.scatter([], [], color="yellow", label="S")       # lege punten om labels al te kunnen tonen
    plt.scatter([], [], color="red", label="I")
    plt.scatter([], [], color="green", label="R")
    plt.legend(loc=0)
    
    plt.show()

In [ ]:
plot_netwerk(X, V)       # knopen en verbindingen van ons netwerk plotten, nog zonder toestanden

#### Attribuez maintenant à chacun des nœuds un état initial.

Initialement, tout le monde est dans l’état $S$, sauf cinq personnes choisies au hasard qui sont infectées.

In [ ]:
n_inf = 5  # initieel aantal geïnfecteerden

#lijst maken van initiële toestanden 
initiele_toestanden = ["S"] * n         # lijst maken van 200 S'n
initiele_toestanden[0: n_inf] = ["I"] * n_inf  # 5 S'n vervangen door I, maakt niet uit welke

In [ ]:
print(initiele_toestanden)
print(len(initiele_toestanden))

In [ ]:
plot_netwerk(X, V, initiele_toestanden)     # knopen en verbindingen van ons netwerk plotten, nu met initiële toestanden

#### Transition d'un état à un autre

Vous avez donc besoin d'une fonction qui, à chaque fois, convertit l'état à l'instant $t$ en l'état à l'instant $t+1$. C'est une fonction assez complexe ! On appelle la transition entre les instants un *pas de temps*.

In [ ]:
def update_toestand(toestanden, V, p_inf=1, p_res=0):
    "Functie die toestand aanpast naar nieuwe toestand per tijdstap."
    n = len(toestanden)        # aantal toestanden is populatiegrootte
    nieuwe_toestanden = []     # maak lijst om de nieuwe toestanden in op te slaan
    
    for i, toestand in enumerate(toestanden):         # ga lijst toestanden af en houd overeenkomstige index bij
        if toestand == "S":                           # persoon i is vatbaar
            # tel aantal geïnfecteerden die persoon i kent
            n_inf_kennissen = 0
            for j in range(n):
                if V[i,j] == 1 and toestanden[j] == "I":     # als persoon i in contact met geïnfecteerde persoon
                    n_inf_kennissen += 1
            # kans dat persoon i ziek wordt door een zieke kennis
            p_ziekte = 1 - (1 - p_inf)**n_inf_kennissen
            # effectief besmet of niet
            if (p_ziekte > np.random.rand()):
                toestand = "I" 
            else:
                toestand = "S"
            nieuwe_toestanden.append(toestand)
        elif toestand == "I":                          # persoon i is vatbaar
            # persoon die geïnfecteerd is, kan resistent worden
            # effectief besmet of niet
            if (p_res > np.random.rand()):
                toestand = "R"  
            else:
                toestand = "I"
            nieuwe_toestanden.append(toestand)
        elif toestand == "R":                          # persoon i is resistent
            # resistente personen blijven resistent
            nieuwe_toestanden.append("R")
    
    return nieuwe_toestanden

In [ ]:
# initiële toestanden updaten voor bepaalde p_inf en p_res voor één tijdstap
p_inf = 0.1
p_res = 0.01

nieuwe_toestanden = update_toestand(initiele_toestanden, V, p_inf, p_res)

print("aantal infecties op t = 0:", 5)
print("aantal infecties op t = 1:", nieuwe_toestanden.count("I"))

In [ ]:
plot_netwerk(X, V, nieuwe_toestanden)         # knopen en verbindingen van ons netwerk plotten, nu met toestanden op t = 1

#### Simulation de l'évolution des états

Vous répétez cela pour toute une série de pas de temps à l'aide d'une boucle for :

In [ ]:
def simuleer_epidemie(init_toestanden, V, tijdstappen, p_inf=1, p_res=0):
    """Simulatie van evolutie toestanden."""
    # sla de toestanden op in een lijst van lijsten
    toestanden_lijst = [init_toestanden]     # lijst huidige toestanden wordt als eerste element in toestanden_lijst gestopt
    toestanden = init_toestanden
    for t in range(tijdstappen):
        toestanden = update_toestand(toestanden, V, p_inf, p_res)
        toestanden_lijst.append(toestanden)
    return toestanden_lijst

Essayez de faire cela pendant 100 pas de temps.

In [ ]:
# simulatie van evolutie toestanden van initiële toestand over 100 tijdstappen
simulatie = simuleer_epidemie(initiele_toestanden, V, 100, p_inf, p_res)   # nog steeds p_inf = 0.1 en p_res = 0.01

Consultez maintenant quelques instantanés au fil du temps (aux pas de temps 0, 10, 20, 50, 70 et 100).

In [ ]:
# verloop na 0, 10, 20, 50, 70 en 100 tijdstappen
for t in [0, 10, 20, 50, 70, 100]:
    toestanden = simulatie[t]             # simulatie is lijst van toestanden van toestanden
    print("tijdstip {}: {} geïnfecteerd, {} resistent".format(t, toestanden.count("I"), toestanden.count("R")))
    plot_netwerk(X, V, toestanden)
    

Vous pouvez suivre plus facilement l'évolution à l'aide d'un graphique. Observez comment les proportions entre les susceptibles, les infectés et les résistants évoluent au fil du temps :

In [ ]:
def plot_progressiekrommen(toestanden_lijst):
    """Evolutie cijfers."""
    tijdstappen = len(toestanden_lijst)     # aantal elementen in toestanden_lijst is gelijk aan aantal tijdstappen
    # tel het aantal personen voor elke toestand per tijdstap
    S = [toestanden.count("S") for toestanden in toestanden_lijst]
    I = [toestanden.count("I") for toestanden in toestanden_lijst]
    R = [toestanden.count("R") for toestanden in toestanden_lijst]
    
    plt.figure()
    
    plt.plot(range(tijdstappen), I, color="purple", label="I")
    plt.plot(range(tijdstappen), S, color="orange", label="S")
    plt.plot(range(tijdstappen), R, color="green", label="R")
    plt.legend(loc=0)
    plt.xlabel("Tijd")
    plt.ylabel("Aantal personen")
    
    plt.show()

In [ ]:
def plot_progressievlakken(toestanden_lijst):
    """Evolutie cijfers."""
    tijdstappen = len(toestanden_lijst)     # aantal elementen in toestanden_lijst is gelijk aan aantal tijdstappen
    # tel het aantal personen voor elke toestand per tijdstap
    S = [toestanden.count("S") for toestanden in toestanden_lijst]
    I = [toestanden.count("I") for toestanden in toestanden_lijst]
    R = [toestanden.count("R") for toestanden in toestanden_lijst]
    
    plt.figure()
    
    plt.stackplot(range(tijdstappen), I, S, R,
                    labels=["I", "S", "R"], colors=["red", "yellow", "lightgreen"])
    plt.legend(loc=0)
    plt.xlabel("Tijd")
    plt.ylabel("Aantal personen")
    
    plt.show()

In [ ]:
plot_progressiekrommen(simulatie)

In [ ]:
plot_progressievlakken(simulatie)

> **Exercice 3**: Si trop de personnes tombent malades trop rapidement, le système de santé peut être submergé, avec des conséquences catastrophiques ! Pour éviter cela, on applique le principe de la *distanciation sociale* : les personnes doivent éviter autant que possible les contacts sociaux. Ainsi, la maladie se transmet plus lentement.- Vous pouvez simuler la distanciation sociale en augmentant $\alpha$, par exemple à 25. Faites-le. Voyez-vous pourquoi le résultat s'appelle l'effet '*flatten the curve*' ?

<div class="alert alert-box alert-info">
Souhaitez-vous télécharger ce notebook, mais le fichier est-il devenu trop volumineux à cause des graphiques ?<br>Supprimez d'abord la sortie des cellules en choisissant dans le menu <b>Cell > All output > Clear</b>.Vous pouvez également enregistrer le notebook au format PDF ou l’imprimer, comme vous le feriez avec une page web.</div>

<img src="images/cclic.png" alt="Banner" align="left" width="100"/><br><br>
Ce notebook de M. Stock et F. wyffels pour Dwengo vzw est sous licence selon une <a href="http://creativecommons.org/licenses/by-nc-sa/4.0/">licence Creative Commons Attribution - Pas d'Utilisation Commerciale - Partage dans les Mêmes Conditions 4.0 International</a>.